In [1]:
from pathlib import Path
from matplotlib import pyplot as plt
import warnings
import prism
import pandas as pd
import pickle
from pathlib import Path 
from imagematerials.reporting.reporting import create_iamc_reporting, create_iamc_eol
from imagematerials.reporting import iamc_config as CFG
from types import SimpleNamespace

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
)
from imagematerials.preprocessing import get_preprocessing_data

from imagematerials.buildings.preprocessing.floorspace import get_image_floorspace, get_floorspace_urban_rural
from imagematerials.buildings.preprocessing.population import compute_population

In [2]:
DESIRED_SCENARIOS = [
    "SSP2", 
    "SSP2_narrow_act",
    "SSP2_narrow_prod",
    "SSP2_narrow",
    "SSP2_slow",
    "SSP2_close",
    "SSP2_narrow_slow_close",
    # "SSP2_19",
    # "SSP2_narrow_19",
    # "SSP2_slow_19",
    # "SSP2_close_19",
    # "SSP2_narrow_slow_close_19",
    # "SSP2_26",
    # "SSP2_narrow_slow_close_26",
    ]  

In [3]:
pkl_path = Path("..","data", "output","all_output_30_jan_26.pkl")

with open(pkl_path, "rb") as f:
    all_output = pickle.load(f)

outdir = Path("..",'data','output','reporting','intermediate','Feb26')

In [4]:
output_vhc = {
    iamc_label: SimpleNamespace(
        vehicles={
            "inflow_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow_materials"][0],
            "stock_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["stock_by_cohort_materials"][0],
            "outflow_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow_by_cohort_materials"][0],
            "inflow":  all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow"][0],
            "stocks":  all_output[CFG.SCENARIO_MAP[iamc_label]]["stocks"][0],
            "outflow": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow"][0],
        }
    )
    for iamc_label in DESIRED_SCENARIOS
}

In [ ]:

# Vehicles reporting
df_vhc = create_iamc_reporting(
    models=output_vhc,
    templates=[
    "Final Product Demand|Transportation|{Vehicle Types}",
    "Product Stock|Transportation|{Vehicle Types}",
    "Stock Retirement|Transportation|{Vehicle Types}",
    "Final Material Demand|{Engineered Material}|{Demand Sector Disaggregated}",
    "Material Stock|{Engineered Material}|{Demand Sector Disaggregated}",
    "Material Outflow|{Engineered Material}|{Demand Sector Disaggregated}",
    ],
    sector="vehicles",               
    model_name="IMAGE 3.4",
    outdir=outdir
)

In [ ]:
output_bld = {
    iamc_label: SimpleNamespace(
        buildings={
            "inflow_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow_materials"][1],
            "stock_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["stock_by_cohort_materials"][1],
            "outflow_by_cohort_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow_by_cohort_materials"][1],
            "inflow":  all_output[CFG.SCENARIO_MAP[iamc_label]]["inflow"][1],
            "stock_by_cohort":  all_output[CFG.SCENARIO_MAP[iamc_label]]["stocks"][1],  # Map stocks to stock_by_cohort
            "outflow_by_cohort": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow"][1],  # Map outflow to outflow_by_cohort
            "stocks":  all_output[CFG.SCENARIO_MAP[iamc_label]]["stocks"][1],
            "outflow": all_output[CFG.SCENARIO_MAP[iamc_label]]["outflow"][1],
        }
    )
    for iamc_label in DESIRED_SCENARIOS
}

# Buildings reporting
df_bld = create_iamc_reporting(
    models = output_bld,
    templates=[
        "Final Product Demand|Buildings|{Building Types}",
        "Product Stock|Buildings|{Building Types}",
        "Stock Retirement|Buildings|{Building Types}",
        "Final Material Demand|{Engineered Material}|{Demand Sector Disaggregated}",
        "Material Stock|{Engineered Material}|{Demand Sector Disaggregated}",
        "Material Outflow|{Engineered Material}|{Demand Sector Disaggregated}",
    ],
    sector = "buildings",
    model_name="IMAGE 3.4",
    outdir=outdir
)


In [6]:
import warnings
from pint.errors import UnitStrippedWarning

output_eol = {
    iamc_label: SimpleNamespace(
        eol={
            "sum_outflow": all_output[CFG.SCENARIO_MAP[iamc_label]]["sum_outflow"],
            #"collected_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["collected_materials"],
            "reusable_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["reusable_materials"],
            "recyclable_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["recyclable_materials"],
            #"losses_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["losses_materials"],
            #"virgin_materials": all_output[CFG.SCENARIO_MAP[iamc_label]]["virgin_materials"],
        }
    )
    for iamc_label in DESIRED_SCENARIOS
}

# End-of-life reporting
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UnitStrippedWarning)

    df_eol = create_iamc_eol(
        models = output_eol,
        templates=[
            # "Material Losses|{Engineered Material}",
            # "Final Material Demand|{Engineered Material}|{Demand Sector Disaggregated}|Reused",
            # "Final Material Demand|{Engineered Material}|{Demand Sector Disaggregated}|Recycled",
            # "Final Material Demand|{Engineered Material}|{Demand Sector Disaggregated}|Virgin Input",
            "Scrap|Iron and Steel|Steel|{Demand Sector Disaggregated}",
            "Scrap|Non-Ferrous Metals|Aluminum|{Demand Sector Disaggregated}",
            "Scrap|Iron and Steel|Steel",
            "Scrap|Non-Ferrous Metals|Aluminum"

        ],
        sector = "eol",
        model_name="IMAGE 3.4",
        outdir = outdir,
    )


No EoL rows produced for SSP2.
No EoL rows produced for SSP2_narrow_act.
No EoL rows produced for SSP2_narrow_prod.
No EoL rows produced for SSP2_narrow.
No EoL rows produced for SSP2_slow.
No EoL rows produced for SSP2_close.
No EoL rows produced for SSP2_narrow_slow_close.


In [ ]:
import warnings
import pandas as pd
from pint.errors import UnitStrippedWarning

warnings.filterwarnings("ignore", category=UnitStrippedWarning)

output_dir = Path("..","data","output","TIMER_input")
output_dir.mkdir(exist_ok=True)

materials = ["Steel", "Concrete"]
common_vars = ["inflow_materials", "outflow_by_cohort_materials"]
eol_vars = ["sum_outflow", "reusable_materials", "recyclable_materials"]
variable_units = {
    "inflow_materials": "Mt",
    "outflow_by_cohort_materials": "Mt",
    "reusable_materials": "Mt",
    "recyclable_materials": "Mt"
}
sector_unit_conversion = {
    "vehicles": {"factor": 1e-9, "unit": "Mt"},             # kg to Mt
    "buildings": {"factor": 1e-9, "unit": "Mt"},            # kg to Mt
    "eol": {"factor": 1e-9, "unit": "Mt"}               # kg to Mt       
                                                            
}

all_rows = []


for scen_id, outputs in all_output.items():
    print(f"Processing {scen_id}")
    rows = []

    for mat in materials:
        # vehicles and buildings
        for var in common_vars:
            for i, sec in enumerate(["vehicles", "buildings"]):
                da = outputs[var][i].to_array().sel(material=mat).sum(dim="Type")
                df = da.to_dataframe(name="value").reset_index()
                # unit conversion
                conv = sector_unit_conversion[sec]
                df["value"] = df["value"] * conv["factor"]
                df["unit"] = conv["unit"]
                df["material"] = mat
                df["sector"] = sec
                df["variable"] = var
                df["scenario"] = scen_id
                rows.append(df)

        # eol 
        for var in eol_vars:
            da = outputs[var].to_array().sel(material=mat).sum(dim="Type")
            df = da.to_dataframe(name="value").reset_index()
            df["value"] = df["value"] * sector_unit_conversion["eol"]["factor"]
            df["unit"] = sector_unit_conversion["eol"]["unit"]
            df["material"] = mat
            df["sector"] = "eol"
            df["variable"] = var
            df["scenario"] = scen_id
            rows.append(df)

    all_rows.extend(rows)
    print(f"Finished {scen_id}")

# Combine all rows into a single DataFrame and pivot it
full_df = pd.concat(all_rows, ignore_index=True)
pivot_df = full_df.pivot_table(
    index=["scenario", "Region", "material", "variable", "unit", "sector"],
    columns="time",  
    values="value"
).reset_index()

pivot_df.to_csv(output_dir / "IMAGE_materials_to_TIMER.csv", index=False)